# Tutorial 5-3: Model Selection and the Bias-Variance Tradeoff
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

--- 
## Objective
A model that performs perfectly on training data but fails on new data is useless. In this final regression tutorial, we explore the 'Science of Evaluation'. We will cover:
1. **Data Splitting:** Properly partitioning data into Training, Validation, and Test sets.
2. **Overfitting vs. Underfitting:** Visualizing how model complexity (e.g., polynomial degree) affects performance.
3. **The Bias-Variance Tradeoff:** Understanding the components of generalization error.
4. **Learning Curves:** Diagnosing whether a model needs more data or a different architecture.

## 1. The Gold Standard: Train/Validation/Test Splits
As discussed in class, we cannot use the same data to train a model and evaluate it. 
- **Training Set:** Used to fit the parameters ($\theta$).
- **Validation Set:** Used to tune hyperparameters (like the degree of a polynomial or the regularization strength $\lambda$).
- **Test Set:** Used only once at the very end to estimate the 'real-world' generalization error.

Let's split a non-linear synthetic dataset.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Generate non-linear data: y = 0.5*x^2 + x + 2 + noise
np.random.seed(42)
m = 100
X = 6 * np.random.rand(m, 1) - 3
y = 0.5 * X**2 + X + 2 + np.random.randn(m, 1)

# First split: Training vs. 'Remainder' (Validation + Test)
X_train, X_rem, y_train, y_rem = train_test_split(X, y, test_size=0.4, random_state=42)

# Second split: Divide the remainder into Validation and Test sets equally
X_val, X_test, y_val, y_test = train_test_split(X_rem, y_rem, test_size=0.5, random_state=42)

print(f"Dataset Sizes: Train ({len(X_train)}), Val ({len(X_val)}), Test ({len(X_test)})")

## 2. Model Complexity and Overfitting
We will now fit three models of increasing complexity to the same data:
1. **Linear (Degree 1):** Too simple to capture the curve (High Bias).
2. **Quadratic (Degree 2):** Matches the true underlying distribution.
3. **High-Polynomial (Degree 15):** Captures many random noise points (High Variance).

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def plot_poly_regression(degree, style, label):
    poly_features = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly_train = poly_features.fit_transform(X_train)
    
    reg = LinearRegression()
    reg.fit(X_poly_train, y_train)
    
    # Predict over a smooth range for visualization
    X_new = np.linspace(-3, 3, 100).reshape(100, 1)
    X_new_poly = poly_features.transform(X_new)
    y_new = reg.predict(X_new_poly)
    
    plt.plot(X_new, y_new, style, label=label, linewidth=2)
    return reg

plt.figure(figsize=(10, 6))
plt.scatter(X_train, y_train, color='gray', alpha=0.5, label='Training Data')
plot_poly_regression(1, 'b--', 'Degree 1 (Underfit)')
plot_poly_regression(2, 'g-', 'Degree 2 (Good Fit)')
plot_poly_regression(20, 'r-', 'Degree 20 (Overfit)')

plt.axis([-3, 3, 0, 10])
plt.legend()
plt.title("Visualizing Overfitting with Polynomials (Slide 39)")
plt.show()

## 3. The Bias-Variance Tradeoff Curve
By calculating the error on both the **Training** and **Validation** sets across different degrees of complexity, we can identify the 'sweet spot'.

- **Low Complexity:** Both errors are high (Underfitting).
- **High Complexity:** Training error is very low, but Validation error explodes (Overfitting).
- **Optimal Complexity:** The point where Validation error is minimized.

In [ ]:
degrees = range(1, 16)
train_errors, val_errors = [], []

for degree in degrees:
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly_train = poly.fit_transform(X_train)
    X_poly_val = poly.transform(X_val)
    
    reg = LinearRegression().fit(X_poly_train, y_train)
    
    train_errors.append(mean_squared_error(y_train, reg.predict(X_poly_train)))
    val_errors.append(mean_squared_error(y_val, reg.predict(X_poly_val)))

plt.figure(figsize=(10, 5))
plt.plot(degrees, train_errors, "b-s", label="Training Error")
plt.plot(degrees, val_errors, "r-^", label="Validation Error")
plt.yscale('log') # Log scale helps see the explosion in error
plt.xlabel("Polynomial Degree (Complexity)")
plt.ylabel("Mean Squared Error")
plt.title("Bias-Variance Tradeoff Curve")
plt.legend()
plt.grid(True, which="both", ls="-", alpha=0.5)
plt.show()

## 4. Diagnosing with Learning Curves
A learning curve plots error against the **number of training examples**. 
- **High Bias (Underfitting):** Adding more data won't help; the error levels off quickly at a high value for both sets.
- **High Variance (Overfitting):** There is a large gap between training and validation error. Adding more data usually helps close this gap.

In [ ]:
def plot_learning_curves(model, X, y):
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=10)
    train_errors, val_errors = [], []
    for m in range(1, len(X_train)):
        model.fit(X_train[:m], y_train[:m])
        y_train_predict = model.predict(X_train[:m])
        y_val_predict = model.predict(X_val)
        train_errors.append(mean_squared_error(y_train[:m], y_train_predict))
        val_errors.append(mean_squared_error(y_val, y_val_predict))

    plt.plot(np.sqrt(train_errors), "r-+", linewidth=2, label="train")
    plt.plot(np.sqrt(val_errors), "b-", linewidth=3, label="val")
    plt.legend()
    plt.xlabel("Training set size")
    plt.ylabel("RMSE")

plt.figure(figsize=(10, 5))
plot_learning_curves(LinearRegression(), X, y)
plt.title("Learning Curves (Diagnosing Bias/Variance)")
plt.axis([0, 80, 0, 3])
plt.show()

## Summary
Selecting the right model requires balancing simplicity and accuracy. 
1. **Validation Sets** are our sandbox for testing different complexities without 'contaminating' our final test results.
2. **The Bias-Variance Tradeoff** is a fundamental law: you can reduce bias by increasing complexity, but only up to the point where variance begins to dominate.
3. **Learning Curves** tell us whether the bottleneck is our data quantity or our model architecture.